# Thực nghiệm 2.5 — Tái thiết kế trực quan hóa Golub et al. (1999)

**Phụ trách:** Thành viên 5 (TV5)  
**Mục tiêu:** Hiện đại hóa các hình trực quan kinh điển trong bài báo tiên phong của Golub et al. (Science, 1999):
- **Fig. 2**: Khám phá lớp không giám sát bằng mạng Self-Organizing Maps (SOM 1x2 và 2x2), đối chiếu với kết quả bài báo và chiếu không gian PCA.
- **Fig. 3A**: (Chờ TV1 tích hợp `GolubWeightedVoting`) Phân bố Prediction Strength (PS) của 50 gen đặc trưng.
- **Fig. 4**: Đánh giá và phân tích lỗi trên tập kiểm tra độc lập (34 bệnh nhân), làm rõ mẫu nghi ngờ (mẫu 66) và các yếu tố gây nhiễu từ nguồn thu thập (loại mô BM/PB, bệnh viện).

Tuân thủ nghiêm ngặt **Style Guide** (`docs/style_guide.md`) và quyết định kiến trúc **ADR-014 / ADR-061**.

In [ ]:
import warnings
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

# Thiet lap duong dan goc du an
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.load import load_golub
from src.evaluate import evaluate, _baseline_pipeline
from src.viz import save_figure
from src.viz_redesign import plot_fig2_som_redesign, plot_fig4_error_analysis

print('Da nap thanh cong cac module tu:', ROOT)

## 1. Nạp dữ liệu tầng Curated và Cleansed

Dữ liệu được đọc qua API chuẩn `load_golub()`:

In [ ]:
X_cu, samples = load_golub('curated')
X_cl, _ = load_golub('cleansed')
print(f'Tang Curated: {X_cu.shape[0]} mau x {X_cu.shape[1]} gen (da loc va log10)')
print(f'Tang Cleansed: {X_cl.shape[0]} mau x {X_cl.shape[1]} gen (thang MAS4)')
display(samples.head())

## 2. Tái thiết kế Fig 2 — Khám phá lớp bằng Self-Organizing Maps (SOM)

Bài báo Golub (1999) sử dụng SOM để phát hiện phân cụm tự nhiên:
- SOM 1x2 tách phần lớn ALL vào A1 (24/27 ALL, 1/11 AML) và AML vào A2 (10/11 AML, 3/27 ALL).
- Bản thiết kế lại cải tiến:
  1. Thêm biểu đồ cột đối chiếu trực quan thành phần cụm bài báo vs kết quả nhóm.
  2. Chiếu 2D trên không gian PCA PC1-PC2 kèm tâm nơ-ron SOM để làm rõ biên phân cụm.
  3. Phân tích SOM 2x2 (4 nơ-ron) để khảo sát phân nhóm phụ B-ALL và T-ALL.

In [ ]:
fig2_report = plot_fig2_som_redesign(X_cu, samples, mode='report')
save_figure(fig2_report, 'F02_som_redesign', mode='report')
plt.show()

fig2_slide = plot_fig2_som_redesign(X_cu, samples, mode='slide')
save_figure(fig2_slide, 'F02_som_redesign', mode='slide')
plt.close(fig2_slide)
print('Da xuat F02_som_redesign cho ca report va slide.')

## 3. Tái thiết kế Fig 4 — Phân tích đánh giá và lỗi trên 34 mẫu kiểm tra độc lập

Tập kiểm tra độc lập gồm 34 bệnh nhân (20 ALL và 14 AML) từ nhiều trung tâm (DFCI, CALGB, CCG, St-Jude):
- Mô hình dự đoán đúng 33/34 mẫu (độ chính xác 97.1%).
- **Ca lâm sàng đặc biệt — Mẫu 66:** Mẫu bệnh nhân AML từ trung tâm CCG (ngân hàng mẫu ung thư trẻ em) bị mô hình dự đoán nhầm thành ALL (xác suất P(AML) chỉ 0.047). Điều này hoàn toàn khớp với bài báo gốc khi mẫu 66 có Prediction Strength chỉ đạt 0.11 (rơi vào vùng không chắc chắn, PS < 0.3).
- **Quản trị dữ liệu (Data Governance):** 5 mẫu lấy từ máu ngoại vi (PB) thay vì tủy xương (BM) vẫn được nhận diện chính xác, cho thấy tính bền vững của các marker tủy/lympho.

In [ ]:
pipe = _baseline_pipeline(k=50)
res_test = evaluate(pipe, X_cl, samples['class'], scheme='original_split')
preds_test = res_test['predictions']

fig4_report = plot_fig4_error_analysis(preds_test, samples, mode='report')
save_figure(fig4_report, 'F04_error_analysis', mode='report')
plt.show()

fig4_slide = plot_fig4_error_analysis(preds_test, samples, mode='slide')
save_figure(fig4_slide, 'F04_error_analysis', mode='slide')
plt.close(fig4_slide)
print('Da xuat F04_error_analysis cho ca report va slide.')

## 4. Kết luận phân tích

1. **SOM (Fig. 2):** Khẳng định tính tự phân cụm mạnh mẽ giữa tế bào tủy (AML) và lympho (ALL). Tuy nhiên, phân nhóm T-ALL vs B-ALL không thể phân tách tự nhiên ở cấu trúc 2x2 nếu không có bước chọn biến giám sát (ủng hộ H5 ở mức ALL/AML, bác bỏ ở mức subtype).
2. **Lỗi phân loại (Fig. 4):** Mẫu 66 là trường hợp duy nhất bị phân loại sai trên cả phân cụm không giám sát (K-Means ở Task 1.5) và mô hình phân loại giám sát (Task 2.4/2.5). Kết hợp góc nhìn quản trị dữ liệu, đây là mẫu tủy xương từ ngân hàng CCG với tỷ lệ blast thấp, đòi hỏi cảnh báo lâm sàng dạng *Uncertain/Review* thay vì ép buộc phân loại tự động.